[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync.ipynb)

**Run order (do these once, then repeat step 3 after every work session)**

| Step | What | Notebook |
|---|---|---|
| 0 | Create a GitHub fine-grained token for this repo only (Contents + Workflows read/write, Actions read). In Colab: key icon (Secrets) > add `GITHUB_TOKEN` > turn on Notebook access. Never paste it in a cell or in chat. | none |
| 1 | Set up CI/CD once: writes `.github/workflows`, dry-runs lint and tests, pushes, reads run status. | [01 CICD](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_01_CICD_Pipeline.ipynb) |
| 2 | First sync as a dry run: set `DRY_RUN = True` in the config cell and check the staged file list. | [00 Sync](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync.ipynb) |
| 3 | Real sync: `DRY_RUN = False`. Run top to bottom; the last cell pushes. | [00 Sync](https://colab.research.google.com/github/Subhash-Pal/Rodic-infra-Colab/blob/main/notebooks/Rodic_00_GitHub_Sync.ipynb) |

Within each notebook, run cells top to bottom (Runtime > Run all is fine). The last cell is always the push.

# Rodic_00_GitHub_Sync
Pushes everything your working Colab produced to `Subhash-Pal/Rodic-infra-Colab`, into one `Vx/` folder.

**How to use**
1. Add Colab Secret `GITHUB_TOKEN` (key icon, left sidebar, enable Notebook access). Fine-grained PAT: Contents read/write on this repo only. Add Workflows read/write if you also push `.github/workflows/` (used by notebook 01).
2. In your working Colab, write code to `/content/work` (e.g. `%%writefile /content/work/module.py`).
3. Give each working cell a markdown heading starting with `### ` that says what it adds. Those headings become the commit message and `CHANGELOG.md`.
4. Run this notebook top to bottom. The last cell is the push.

Safety: secret scan, 50 MB file guard, optional pytest gate, dry-run switch. Only `VERSION_DIR/` and `notebooks/` are staged.

### Config and authentication helpers
Defines repo, target folder, and a `git()` wrapper that sends the token as a header, never stores it in the remote URL, and masks it in output.

In [10]:
import os, re, json, base64, shutil, subprocess, pathlib, datetime
from google.colab import userdata

# ---------- CONFIG: edit only this block ----------
REPO         = "Subhash-Pal/Rodic-infra-Colab"
BRANCH       = "main"
VERSION_DIR  = "v4"                 # folder inside the repo that receives your work (Vx/ rule)
WORK_DIR     = "/content/work"      # folder in THIS Colab runtime whose files get pushed
NOTEBOOK_PATH = ""                  # optional: Drive path of the Colab you are editing, e.g.
                                    # "/content/drive/MyDrive/Colab Notebooks/Rodic_Edge_CV.ipynb"
RUN_TESTS    = True                 # run pytest before pushing, if tests exist
DRY_RUN      = False                # True = stage + show diff, no commit/push
MAX_FILE_MB  = 50
# --------------------------------------------------

REPO_DIR = "/content/_repo"
TOKEN = userdata.get("GITHUB_TOKEN-gemini-resilient")
_B64 = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
AUTH = ["-c", f"http.extraheader=AUTHORIZATION: basic {_B64}"]

def _mask(t): return t.replace(TOKEN, "***").replace(_B64, "***")

def git(*args, cwd=REPO_DIR, auth=False, check=True):
    cmd = ["git"] + (AUTH if auth else []) + list(args)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = _mask(r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

### Clone or update the repo
Fresh clone on first run, otherwise fetch and rebase so your push does not collide with remote changes.

In [6]:
if not os.path.isdir(f"{REPO_DIR}/.git"):
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR, cwd="/content", auth=True)
else:
    git("fetch", "origin", BRANCH, auth=True)
    git("checkout", BRANCH)
    git("pull", "--rebase", "origin", BRANCH, auth=True)
git("config", "user.name", "Subhash Pal (Colab)")
git("config", "user.email", "colab-sync@users.noreply.github.com")
print(git("log", "--oneline", "-5"))

dd54e30 Created using Colab
a4ab6e5 notebooks: move all four notebooks into notebooks/ and fix Colab badge paths
af1f771 notebooks: add Open in Colab badges and run order
07ef6c0 notebooks: add Colab GitHub-sync and CI/CD pipeline notebooks
5f45ea0 Created using Colab


### Secret scanner
Blocks the push if any file contains a Gemini/Google key, GitHub token, Anthropic key or private key block. Relevant because your six `GEMINI_API_KEY*` values must never reach a public repo.

In [7]:
SECRET_PATTERNS = {
    "Google/Gemini API key": r"AIza[0-9A-Za-z_\-]{35}",
    "GitHub token":          r"(ghp_|gho_|ghs_)[A-Za-z0-9]{36}|github_pat_[A-Za-z0-9_]{50,}",
    "Anthropic key":         r"sk-ant-[A-Za-z0-9_\-]{20,}",
    "Private key block":     r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
}
TEXT_EXT = {".py",".md",".txt",".yaml",".yml",".json",".ipynb",".toml",".ini",".cfg",".env",".sh",".csv"}

def scan_for_secrets(root):
    hits = []
    for p in pathlib.Path(root).rglob("*"):
        if p.is_file() and (p.suffix.lower() in TEXT_EXT or p.name.startswith(".env")):
            try: txt = p.read_text(errors="ignore")
            except Exception: continue
            for label, pat in SECRET_PATTERNS.items():
                if re.search(pat, txt):
                    hits.append((str(p), label))
    return hits
print("Secret scanner ready.")

Secret scanner ready.


### Stage working files into the version folder
Copies `WORK_DIR` into `VERSION_DIR/`, ignoring caches, `.env` and key files, dropping files over the size limit, and optionally copying the working notebook into `notebooks/`.

In [11]:
IGNORE = shutil.ignore_patterns("__pycache__", "*.pyc", ".ipynb_checkpoints", ".env", ".env.*",
                                "*.key", "*.pem", "sample_data", ".git", "venv", ".venv")
dest = pathlib.Path(REPO_DIR) / VERSION_DIR
dest.mkdir(parents=True, exist_ok=True)
src = pathlib.Path(WORK_DIR)
assert src.is_dir(), f"WORK_DIR not found: {WORK_DIR}"

skipped_big = []
for p in src.rglob("*"):
    if p.is_file() and p.stat().st_size > MAX_FILE_MB * 1024 * 1024:
        skipped_big.append(str(p))
shutil.copytree(src, dest, ignore=IGNORE, dirs_exist_ok=True)
for b in skipped_big:                       # never push huge weights/datasets by accident
    rel = pathlib.Path(b).relative_to(src)
    (dest / rel).unlink(missing_ok=True)

if NOTEBOOK_PATH and os.path.exists(NOTEBOOK_PATH):
    nbdir = pathlib.Path(REPO_DIR) / "notebooks"
    nbdir.mkdir(exist_ok=True)
    shutil.copy(NOTEBOOK_PATH, nbdir / pathlib.Path(NOTEBOOK_PATH).name)

print(f"Staged {WORK_DIR} -> {VERSION_DIR}/")
if skipped_big:
    print(f"Skipped (> {MAX_FILE_MB} MB, use Releases/Drive/HF instead):", *skipped_big, sep="\n  ")

AssertionError: WORK_DIR not found: /content/work

### Build changelog and commit message from cell headings
Reads every `### ` markdown heading in your working notebook and records it as a dated CHANGELOG entry.

In [ ]:
def headings_from_notebook(path):
    """Collect '### ...' markdown headings = what each cell added."""
    if not path or not os.path.exists(path): return []
    nbj = json.load(open(path))
    out = []
    for c in nbj.get("cells", []):
        if c["cell_type"] == "markdown":
            first = "".join(c["source"]).strip().splitlines()[:1]
            if first and first[0].startswith("###"):
                out.append(first[0].lstrip("# ").strip())
    return out

added = headings_from_notebook(NOTEBOOK_PATH)
stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=5, minutes=30))).strftime("%Y-%m-%d %H:%M IST")
if added:
    cl = pathlib.Path(REPO_DIR) / VERSION_DIR / "CHANGELOG.md"
    old = cl.read_text() if cl.exists() else f"# Changelog ({VERSION_DIR})\n"
    entry = f"\n## {stamp}\n" + "\n".join(f"- {h}" for h in added) + "\n"
    cl.write_text(old + entry)
    print(entry)
else:
    print("No '### ' headings found (set NOTEBOOK_PATH to auto-build the changelog).")
COMMIT_MSG = f"{VERSION_DIR}: " + ("; ".join(added)[:180] if added else f"colab sync {stamp}")

### Test gate
Runs pytest on the staged folder if tests exist. A failing test blocks the push.

In [ ]:
tests_dir = pathlib.Path(REPO_DIR) / VERSION_DIR
has_tests = any(tests_dir.rglob("test_*.py")) or any(tests_dir.rglob("*_test.py"))
if RUN_TESTS and has_tests:
    subprocess.run("pip -q install pytest pyyaml scikit-image scipy", shell=True)
    r = subprocess.run(["python", "-m", "pytest", "-q", str(tests_dir)], cwd=REPO_DIR, capture_output=True, text=True)
    print(r.stdout[-2500:])
    TESTS_OK = (r.returncode == 0)
else:
    print("Tests skipped (disabled or none found).")
    TESTS_OK = True
print("TESTS_OK =", TESTS_OK)

### FINAL CELL: commit and push
Runs the secret scan and test gate, then commits, rebases and pushes. Set `DRY_RUN = True` in the config cell to preview only.

In [ ]:
hits = scan_for_secrets(REPO_DIR + "/" + VERSION_DIR) + (scan_for_secrets(REPO_DIR + "/notebooks") if os.path.isdir(REPO_DIR + "/notebooks") else [])
if hits:
    print("BLOCKED: possible secrets found. Remove them, rotate the key, then re-run.")
    for f, l in hits: print("  ", l, "->", f)
elif not TESTS_OK:
    print("BLOCKED: tests failed. Fix them or set RUN_TESTS=False knowingly.")
else:
    git("add", "-A", VERSION_DIR)
    if os.path.isdir(REPO_DIR + "/notebooks"): git("add", "-A", "notebooks")
    status = git("status", "--short")
    if not status:
        print("Nothing changed - nothing to push.")
    elif DRY_RUN:
        print("DRY RUN - would commit:\n" + status)
    else:
        git("commit", "-m", COMMIT_MSG)
        git("pull", "--rebase", "origin", BRANCH, auth=True)
        git("push", "origin", BRANCH, auth=True)
        print("Pushed:", git("log", "--oneline", "-1"))
        print(f"https://github.com/{REPO}/tree/{BRANCH}/{VERSION_DIR}")